## Data preprocessing
In the following codeblocks, we will preprocess the date for the fine-tuning of GLiNER2. These are the steps we will follow:
1. Filter the Label Studio annotations (only the most recent and validated annotations)
2. Reduce the dataset (optional, to speed up training)
3. Convert the Label Studio annotations to GLiNER2 training data format
4. split the data into training, validation and test sets

### 1. filter the Label Studio annotations

Label Studio only allows to output all the annotations (so also the different annotations for the same text). Firt you need to make sure that you only have those annotations that you need to train the model. In this code snippet, we filter the annotations to only include those that are relevant to us (the most recent and validated annotations). We do this by filtering on the text "id" and "annotator".

In [3]:
import json

# Load your JSON data from the specified input file
with open("LS_Labeled_BHF.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Define ID ranges
id_ranges = [
    range(11317, 11617)  # 11317–11616  -> last number is exclusive
]

# Define allowed annotators
allowed_annotators = {17, 18, 19}

# Filter the data
filtered_data = [
    item for item in data
    if any(item["id"] in r for r in id_ranges)
    and item.get("annotator") in allowed_annotators
]

# Save the filtered JSON to the specified output file
with open("LS_labeled_BHF_filtered.json", "w", encoding="utf-8") as f:
    json.dump(filtered_data, f, ensure_ascii=False, indent=2)

print(f"Filtered {len(filtered_data)} items out of {len(data)} total.")

Filtered 300 items out of 999 total.


### 2. reduce the dataset (optional)

In [2]:
# reducing the dataset

import json
import random

# Paths
input_path = "LS_labeled_BHF_filtered.json"


# Load the JSON file
with open(input_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# Set the fraction of entries to keep (0.5 = 50%)
fraction_to_keep = 0.5
num_to_keep = int(len(data) * fraction_to_keep)

# Randomly select entries to keep
reduced_data = random.sample(data, num_to_keep)

output_path = f"LS_labeled_BHF_reduced_{fraction_to_keep}.json"

# Save the reduced JSON
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(reduced_data, f, ensure_ascii=False, indent=2)

print(f"Reduced from {len(data)} entries to {len(reduced_data)} entries.")

Reduced from 190 entries to 95 entries.


### 3. Label Studio to GLiNER2 Training Data Conversion
This code is used to convert the human labeled entities in Label Studio to a format that can be used for training GLiNER2. It reads the labeled data from Label Studio, extracts the relevant information, and creates a new JSON file that can be used for training.
It follows the recommended format:
```json
{"input": "text to process", "output": {"schema_definition": "with_annotations"}}
```

In [1]:
import json
from collections import defaultdict


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def convert_to_gliner2_format(annotation_data, schema):
    """
    Converts Label Studio-style annotations to GLiNER2 format.
    Only uses labels defined inside schema["entities"].
    """

    # ✅ Only take actual NER entity types
    entity_types = list(schema["entities"].keys())

    gliner_data = []

    for item in annotation_data:
        text = item["text"]
        annotations = item.get("label", [])

        entities_dict = defaultdict(list)

        # Collect entities
        for ann in annotations:
            entity_text = ann["text"].strip()
            labels = ann.get("labels", [])

            for label in labels:
                # ✅ Only include labels that exist in schema
                if label in entity_types:
                    entities_dict[label].append(entity_text)

        # Deduplicate while preserving order
        for label in entities_dict:
            seen = set()
            deduped = []
            for ent in entities_dict[label]:
                if ent not in seen:
                    deduped.append(ent)
                    seen.add(ent)
            entities_dict[label] = deduped

        # Ensure ALL schema entity types are present (even if empty)
        for entity_type in entity_types:
            if entity_type not in entities_dict:
                entities_dict[entity_type] = []

        formatted_item = {
            "text": text,
            "entities": dict(entities_dict)
            }


        gliner_data.append(formatted_item)

    return gliner_data


def save_json(data, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


if __name__ == "__main__":
    annotation_file = "LS_labeled_BHF_filtered.json"
    schema_file = "gliner_schema_BelHisFirm.json"
    output_file = "gliner2_data_BHF.json"

    annotations = load_json(annotation_file)
    schema = load_json(schema_file)

    gliner_data = convert_to_gliner2_format(annotations, schema)
    save_json(gliner_data, output_file)

    print(f"Converted {len(gliner_data)} examples.")

Converted 300 examples.


### 4. split the data into training and test sets
To be able to evaluate the performance of our model, we need to split our dataset into training (80%) and test (20%) sets. This code snippet takes the converted GLiNER2 training data and splits it into three separate JSON files for training, validation and testing.

In [2]:
!pip install scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 15.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 14.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn] [scikit-learn]


In [18]:
from sklearn.model_selection import train_test_split

# Load the converted GLiNER2 training data
with open("gliner2_data_BHF.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Split into train (80%) and test (20%)
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)


def choose_entity_types_to_train(schema, selected_entities):
    """Return validated entity types chosen by the user.

    selected_entities can contain entity names ("street") or 1-based indices (1, "2").
    """
    entity_types = list(schema["entities"].keys())
    print("Available entity types:")
    for i, et in enumerate(entity_types, start=1):
        print(f"{i}. {et}")

    chosen = []
    for value in selected_entities:
        if isinstance(value, int):
            idx = value - 1
            if 0 <= idx < len(entity_types):
                chosen.append(entity_types[idx])
            else:
                raise ValueError(f"Index {value} is out of range.")
        elif isinstance(value, str) and value.strip().isdigit():
            idx = int(value.strip()) - 1
            if 0 <= idx < len(entity_types):
                chosen.append(entity_types[idx])
            else:
                raise ValueError(f"Index {value} is out of range.")
        elif isinstance(value, str):
            if value in entity_types:
                chosen.append(value)
            else:
                raise ValueError(f"Unknown entity type: {value}")
        else:
            raise ValueError(f"Unsupported selection value: {value}")

    # Remove duplicates while preserving order
    return list(dict.fromkeys(chosen))


def filter_dataset_by_entities(dataset, selected_entity_types):
    """Keep only selected entity types in each training example."""
    filtered = []
    for item in dataset:
        filtered_entities = {
            entity_type: item.get("entities", {}).get(entity_type, [])
            for entity_type in selected_entity_types
        }
        filtered.append({"text": item["text"], "entities": filtered_entities})
    return filtered


# Kies hier zelf de entiteiten die je wil trainen (namen of 1-based indices)
selected_entity_types_input = ["street", "housenumber"]

with open("gliner_schema_BelHisFirm.json", "r", encoding="utf-8") as f:
    schema = json.load(f)

selected_entity_types = choose_entity_types_to_train(schema, selected_entity_types_input)
train_data = filter_dataset_by_entities(train_data, selected_entity_types)
test_data = filter_dataset_by_entities(test_data, selected_entity_types)

# Save the filtered splits to separate JSON files
with open("./data/train.json", "w", encoding="utf-8") as f:
    json.dump(train_data, f, indent=2, ensure_ascii=False)
with open("./data/test.json", "w", encoding="utf-8") as f:
    json.dump(test_data, f, indent=2, ensure_ascii=False)

print(f"Selected entity types: {selected_entity_types}")
print(f"Train: {len(train_data)} examples")
print(f"Test: {len(test_data)} examples")

Train: 240 examples
Test: 60 examples


## finetuning GLiNER2 with LoRA
Now that we have the training data in the correct format, we can use it to fine-tune GLiNER2 specifically for our use case.

In [1]:
!pip install gliner2

## Step 1: prepare Domain-Specific Data
We have already prepared our training data in the previous step. The file `train.json` contains the text and corresponding entities in the format required for GLiNER2 training. Now we can load this data and use it for fine-tuning the model.

In [3]:
import json
from gliner2.training.data import InputExample


with open("./data/train.json", "r", encoding="utf-8") as f:         #change the path to your training
    data = json.load(f)



train_data = [
    InputExample(text=item["text"], entities=item["entities"])
    for item in data
]

## Step 2: Set Up LoRA Training
Next, we will set up the LoRA training configuration. This involves specifying the adaptor we want to train, the training parameters and the LoRA settings.

In [15]:
from gliner2 import GLiNER2
from gliner2.training.trainer import GLiNER2Trainer, TrainingConfig

# LoRA configuration
config = TrainingConfig(
    output_dir="./BelHisFirm_adaptor",       # change this to your desired output directory
    experiment_name="BelHisFirm",            # change this to your desired experiment name

    # Training parameters
    num_epochs=20,
    batch_size=4,
    gradient_accumulation_steps=4,
    encoder_lr=1e-5,
    task_lr=5e-4,

    # LoRA settings
    use_lora=True,                              # Enable LoRA
    lora_r=4,                                   # Rank (4, 8, 16, 32)
    lora_alpha=16.0,                           # Scaling factor (usually 2*r)
    lora_dropout=0.0,                          # Dropout for LoRA layers
    lora_target_modules=["encoder"],           # Apply to all encoder layers (query, key, value, dense)
    save_adapter_only=True,                    # Save only adapter (not full model)

    # Optimization
    eval_strategy="epochs",  # Evaluates and saves at end of each epoch
    eval_steps=200,  # Used when eval_strategy="steps"
    logging_steps=50,
    fp16=True,  # Use mixed precision if GPU available
    early_stopping=False,
)

## Step 3: Train the LoRA Adaptor
Now we can initialize the GLiNER2 model and the trainer, and start the training process

In [16]:
# Load base model
base_model = GLiNER2.from_pretrained("fastino/gliner2-base-v1")

# Create trainer
trainer = GLiNER2Trainer(model=base_model, config=config)

# Train adapter
trainer.train(train_data=train_data)

# Adapter automatically saved to ./BelHisFirm_adapter/final/

2026-03-23 16:20:54 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/fastino/gliner2-base-v1/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-23 16:20:54 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/fastino/gliner2-base-v1/283f4af5e598631a5352b8c388b6906853146f07/config.json "HTTP/1.1 200 OK"
2026-03-23 16:20:54 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/fastino/gliner2-base-v1/resolve/main/encoder_config/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-23 16:20:54 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/fastino/gliner2-base-v1/283f4af5e598631a5352b8c388b6906853146f07/encoder_config%2Fconfig.json "HTTP/1.1 200 OK"
2026-03-23 16:20:54 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/fastino/gliner2-base-v1/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-23 16:20:54 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/api/r

🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first


2026-03-23 16:20:56 - INFO - gliner2.training.trainer - Using device: cuda
2026-03-23 16:20:56 - INFO - gliner2.training.trainer - Setting up LoRA for parameter-efficient fine-tuning...
2026-03-23 16:20:56 - INFO - gliner2.training.trainer - Froze all model parameters for LoRA training
2026-03-23 16:20:56 - INFO - gliner2.training.lora - Applied LoRA to 74 layers
2026-03-23 16:20:56 - INFO - gliner2.training.trainer - LoRA setup complete: 678,916 trainable params out of 209,155,737 total (0.32%)


🔧 LoRA Configuration
Enabled            : True
Rank (r)           : 4
Alpha              : 16.0
Scaling (α/r)      : 4.0000
Dropout            : 0.0
Target modules     : encoder, classifier
LoRA layers        : 74
----------------------------------------------------------------------
Trainable params   : 678,916 / 209,155,737 (0.32%)
Memory savings     : ~99.7% fewer gradients


Validating records: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 240/240 [00:00<00:00, 11150.24record/s]
2026-03-23 16:20:56 - INFO - gliner2.training.trainer - Optimizer: LoRA params only = 148, LR=0.0005
/home/hbekaer/PycharmProjects/ghentcdh-glinerv2-tutorial/.venv/lib/python3.13/site-packages/gliner2/training/trainer.py:900: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler(enabled=self.config.fp16)
2026-03-23 16:20:56 - INFO - gliner2.training.trainer - ***** Running Training *****
2026-03-23 16:20:56 - INFO - gliner2.training.trainer -   Num examples = 240
2026-03-23 16:20:56 - INFO - gliner2.training.trainer -   Num epochs = 20
2026-03-23 16:20:56 - INFO - gliner2.training.trainer -   

Training:   0%|          | 0/300 [00:00<?, ?it/s]

/home/hbekaer/PycharmProjects/ghentcdh-glinerv2-tutorial/.venv/lib/python3.13/site-packages/gliner2/training/trainer.py:947: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp, dtype=amp_dtype):
/home/hbekaer/PycharmProjects/ghentcdh-glinerv2-tutorial/.venv/lib/python3.13/site-packages/gliner2/training/trainer.py:993: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  self.scheduler.step()
2026-03-23 16:20:59 - WARNING - gliner2.training.trainer - OOM at step 13, batch skipped. Consider reducing batch_size or max sequence length.
2026-03-23 16:21:00

KeyboardInterrupt: 

In [17]:
def evaluate_adapter(model, adapter_path, test_data):
    """Evaluate adapter performance on test data."""
    model.load_adapter(adapter_path)

    results = {
        "total": 0,
        "correct": 0,
        "precision_sum": 0,
        "recall_sum": 0,
    }

    for sample in test_data:
        pred = model.extract_entities(sample["text"], sample["entity_types"])

        # Compute metrics
        metrics = compute_metrics(pred['entities'], sample["true_entities"])
        results["total"] += 1
        results["precision_sum"] += metrics["precision"]
        results["recall_sum"] += metrics["recall"]

    avg_precision = results["precision_sum"] / results["total"]
    avg_recall = results["recall_sum"] / results["total"]
    f1 = 2 * avg_precision * avg_recall / (avg_precision + avg_recall)

    return {
        "precision": avg_precision,
        "recall": avg_recall,
        "f1": f1,
        "samples": results["total"]
    }

ImportError: cannot import name 'compute_metrics' from 'sklearn' (/home/hbekaer/PycharmProjects/ghentcdh-glinerv2-tutorial/.venv/lib/python3.13/site-packages/sklearn/__init__.py)

## Step 4: evaluate the trained adapter
After training, we can evaluate the performance of our trained adapter on the validation and test sets. We will load the trained adapter, run it on the validation and test data, and calculate evaluation metrics such as precision, recall and F1-score for the extracted entities. This will help us understand how well our adapter is performing on unseen data and whether it has successfully learned to extract the relevant entities for our specific use case.

In [6]:
## Load the trained adapter

model = GLiNER2.from_pretrained("fastino/gliner2-base-v1")
model.load_adapter("./BelHisFirm_adaptor/final")

# Check if adapter is loaded
if model.has_adapter:
    print("Adapter is loaded")

# Get adapter configuration
config = model.adapter_config
print(f"LoRA rank: {config.lora_r}")

2026-03-23 14:50:56 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/fastino/gliner2-base-v1/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-23 14:50:56 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/fastino/gliner2-base-v1/283f4af5e598631a5352b8c388b6906853146f07/config.json "HTTP/1.1 200 OK"
2026-03-23 14:50:56 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/fastino/gliner2-base-v1/resolve/main/encoder_config/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-23 14:50:56 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/fastino/gliner2-base-v1/283f4af5e598631a5352b8c388b6906853146f07/encoder_config%2Fconfig.json "HTTP/1.1 200 OK"
2026-03-23 14:50:56 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/fastino/gliner2-base-v1/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-23 14:50:56 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/api/r

🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first
Adapter is loaded
LoRA rank: 8


In [8]:
from gliner_to_labelstudio import (
    load_gliner_schema_config,
    create_gliner_schema_from_config_file,
)
SCHEMA_CONFIG_PATH = "./gliner_schema_BelHisFirm.json"
schema_config = load_gliner_schema_config(SCHEMA_CONFIG_PATH)

extractor = model

schema = create_gliner_schema_from_config_file(extractor, SCHEMA_CONFIG_PATH)

output_path = "gliner2_evaluation_BelHisFirm_trained.json"

# run on validation set
with open("./data/val.json", "r", encoding="utf-8") as f:
    val_data = json.load(f)
results_val = []
for item in val_data:
    text = item["text"]
    result = extractor.extract(text, schema, threshold=0.1, include_confidence=True, include_spans=True, format_results=False)
    results_val.append(result)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results_val, f, indent=2, ensure_ascii=False)

print(f"✓ Evaluation complete! Results saved to {output_path}")

In [10]:
# test the base model
from gliner_to_labelstudio import (
    load_gliner_schema_config,
    create_gliner_schema_from_config_file,
)

model = GLiNER2.from_pretrained("fastino/gliner2-base-v1")
model.unload_adapter()  # Ensure no adapter is loaded

SCHEMA_CONFIG_PATH = "./gliner_schema_BelHisFirm.json"
schema_config = load_gliner_schema_config(SCHEMA_CONFIG_PATH)

extractor = model

schema = create_gliner_schema_from_config_file(extractor, SCHEMA_CONFIG_PATH)

output_path = "gliner2_evaluation_BelHisFirm_base.json"

# run on validation set
with open("./data/val.json", "r", encoding="utf-8") as f:
    val_data = json.load(f)
results_val = []
for item in val_data:
    text = item["text"]
    result = extractor.extract(text, schema, threshold=0.1, include_confidence=True, include_spans=True, format_results=False)
    results_val.append(result)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results_val, f, indent=2, ensure_ascii=False)

print(f"✓ Evaluation complete! Results saved to {output_path}")

2026-03-23 16:00:12 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/fastino/gliner2-base-v1/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-23 16:00:12 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/fastino/gliner2-base-v1/283f4af5e598631a5352b8c388b6906853146f07/config.json "HTTP/1.1 200 OK"
2026-03-23 16:00:12 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/fastino/gliner2-base-v1/resolve/main/encoder_config/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-23 16:00:12 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/fastino/gliner2-base-v1/283f4af5e598631a5352b8c388b6906853146f07/encoder_config%2Fconfig.json "HTTP/1.1 200 OK"
2026-03-23 16:00:12 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/fastino/gliner2-base-v1/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-23 16:00:12 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/api/r

🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first
✓ Evaluation complete! Results saved to gliner2_evaluation_BelHisFirm_base.json


## load and test the trained adapter
After training, we can load the trained adapter and test it on some example sentences to see how

In [17]:
from gliner_to_labelstudio import (
    load_gliner_schema_config,
    create_gliner_schema_from_config_file,
)
SCHEMA_CONFIG_PATH = "./gliner_schema_BelHisFirm.json"
schema_config = load_gliner_schema_config(SCHEMA_CONFIG_PATH)

extractor = model

schema = create_gliner_schema_from_config_file(extractor, SCHEMA_CONFIG_PATH)

text = """
[1] Eo namque tempore, quo fabrica altaris ecclesiæ nostræ agebatur, contigit adesse ibi quemdam Joseph nomine, qui ita curvus erat, ut supra genua incumbens caput sursum erigere non posset: sed nec penitus immobile id uno in loco tenere diutius valebat. Qualiter autem illi acciderit, referebat: quia dum esset solus positus cum suo equo in campo vicini sui, ejusque herbam injuste depasceret, a fulgure ignis divinitus percussus ita exustus est, ut mox capite deposito curvus redderetur.
"""

results = results = extractor.extract(text, schema, threshold=0.1, include_confidence=True, include_spans=True, format_results=False)

print(json.dumps(results, indent=2, ensure_ascii=False))


ModuleNotFoundError: No module named 'gliner_to_labelstudio'